In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\karup\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_csv(r"C:\Users\karup\Downloads\imdb-reviews.csv")

In [3]:
df

,label,text
0,False,I rented I AM CURIOUS-YELLOW from my video sto...
1,False,"""I Am Curious: Yellow"" is a risible and preten..."
2,False,If only to avoid making this type of film in t...
3,False,This film was probably inspired by Godard's Ma...
4,False,"Oh, brother...after hearing about this ridicul..."
...,...,...
24995,True,A hit at the time but now better categorised a...
24996,True,I love this movie like no other. Another time ...
24997,True,This film and it's sequel Barry Mckenzie holds...
24998,True,'The Adventures Of Barry McKenzie' started lif...


In [4]:
df = df.rename(columns={'text': 'review', 'label': 'sentiment'})
# convert True/False → text
df['sentiment'] = df['sentiment'].map({True:'positive', False:'negative'})

### 1. DATA UNDERSTANDING

In [5]:
df.head()
df.info()
df.shape
df.columns
df['sentiment'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB


sentiment
negative    12500
positive    12500
Name: count, dtype: int64

- Dataset has 25,000 samples
- Contains review text and sentiment labels
- Balanced dataset 

### 2. NLP PREPROCESSING

#### Create function

In [6]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    
    return " ".join(words)

#### Apply

In [7]:
df['cleaned_text'] = df['review'].apply(clean_text)

In [8]:
df[['review', 'cleaned_text']].head()

,review,cleaned_text
0,I rented I AM CURIOUS-YELLOW from my video sto...,rent curiou yellow video store controversi sur...
1,"""I Am Curious: Yellow"" is a risible and preten...",curiou yellow risibl pretenti steam pile matte...
2,If only to avoid making this type of film in t...,avoid make type film futur film interest exper...
3,This film was probably inspired by Godard's Ma...,film probabl inspir godard masculin f minin ur...
4,"Oh, brother...after hearing about this ridicul...",oh brother hear ridicul film umpteen year thin...


### 3. FEATURE ENGINEERING

#### Bag of Words (BoW)

In [9]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [10]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_text'])

#### TF-IDF

In [11]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_text'])

In [18]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

### 4. MODEL BUILDING

#### Train-test split

In [13]:
X = df['cleaned_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### TF-IDF for models

In [14]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

#### Train models

In [15]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


### 5. MODEL EVALUATION

#### Evaluation function

In [19]:
def evaluate(model):
    y_pred = model.predict(X_test_tfidf)
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred))
    print("------")

In [20]:
print("Logistic Regression")
evaluate(lr)

print("Naive Bayes")
evaluate(nb)

print("Decision Tree")
evaluate(dt)

Logistic Regression
Accuracy: 0.8834
Precision: 0.8709048361934477
Recall: 0.8985915492957747
F1: 0.8845315904139434
------
Naive Bayes
Accuracy: 0.8546
Precision: 0.8477056962025317
Recall: 0.862374245472837
F1: 0.8549770596449232
------
Decision Tree
Accuracy: 0.7086
Precision: 0.7069243156199678
Recall: 0.7066398390342052
F1: 0.706782048701952
------


### 6. COMPARISON & INSIGHTS

- TF-IDF performed better than Bag of Words as it captures word importance.
- Logistic Regression achieved the highest accuracy and F1-score.
- Naive Bayes performed well and was faster compared to other models.
- Decision Tree showed lower performance and signs of overfitting.
- Preprocessing steps like stopword removal and stemming improved model performance.

### PIPELINE FLOW

Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison